# 01 - Data Loading
Load CMEMS oceanographic data and AIS fishing effort for the West Philippine Sea ConvLSTM pipeline.

**Outputs:** `physics_raw_region.nc`, `bgc_raw_region.nc`, `ais_raw_region.parquet`

Run this notebook once to cache regional data, then proceed to `02_preprocessing.ipynb`.

In [1]:
# CELL: install dependencies
!pip install -q xarray netCDF4 pandas numpy matplotlib pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 42.2 MB/s eta 0:00:00


In [2]:
# CELL: imports, drive mount, and shared pipeline CONFIG
# =========================================================
# All spatial, temporal, path, and modelling parameters live here.
# =========================================================
from google.colab import drive
import xarray as xr
import pandas as pd
import numpy as np
import glob, os

drive.mount('/content/drive')

CONFIG = {
    # ------------------------------------------------------------------
    # Spatial
    # ------------------------------------------------------------------
    # Broad regional bbox: used in THIS notebook to cache raw CMEMS/AIS.
    # Wide enough to allow experimenting with different sub-regions in 02
    # without re-downloading CMEMS data.
    'bbox_regional': {
        'lat_min': 0,   'lat_max': 30,
        'lon_min': 110, 'lon_max': 140,
    },
    # WPS model target bbox: used from Notebook 02 onward.
    # Narrow domain improves class balance and F1 for ConvLSTM training.
    'bbox_model': {
        'lat_min': 10,  'lat_max': 20,
        'lon_min': 114, 'lon_max': 120,
    },

    # ------------------------------------------------------------------
    # Temporal
    # ------------------------------------------------------------------
    # Full date range for raw CMEMS/AIS load (this notebook).
    'date_full': {'start': '2014-01-01', 'end': '2024-12-31'},
    # ConvLSTM training window (Notebook 02 onward).
    # Starts 2019 to use the denser post-2019 AIS coverage.
    'date_model': {'start': '2019-01-01', 'end': '2024-12-31'},

    # ------------------------------------------------------------------
    # Paths
    # ------------------------------------------------------------------
    'data_dir': '/content/drive/MyDrive/fishing_project/',
    'files': {
        # Source CMEMS NetCDF files (placed in data_dir by the user)
        'physics_w_nc':  'cmems_mod_glo_phy_my_0.083deg_P1M-m_1779636039565.nc',
        'physics_ht_nc': 'cmems_mod_glo_phy_my_0.083deg_P1M-m_1779635380319.nc',
        'bgc_src_nc':    'cmems_mod_glo_bgc_my_0.25deg_P1M-m_1779635372583.nc',
        # Notebook 01 outputs -> Notebook 02 inputs
        'physics_nc':    'physics_raw_region.nc',
        'bgc_nc':        'bgc_raw_region.nc',
        'ais_parquet':   'ais_raw_region.parquet',
        'ais_csv_gz':    'ais_raw_region.csv.gz',
        # Notebook 02 outputs -> Notebook 03 inputs
        'ais_gridded_nc':  'ais_fishing_effort_gridded.nc',
        'preprocessed_nc': 'preprocessed_features.nc',
    },

    # ------------------------------------------------------------------
    # AIS
    # ------------------------------------------------------------------
    # Columns to read from each GFW monthly CSV (skips vessel metadata)
    'ais_use_cols': ['date', 'cell_ll_lat', 'cell_ll_lon', 'fishing_hours'],

    # ------------------------------------------------------------------
    # Depth selection for CMEMS variables
    # ------------------------------------------------------------------
    'physics_surface_depth': 0.49,   # metres, nearest-neighbour selection
    'bgc_depth_range': (0.51, 5.14), # metres, averaged over this range

    # ------------------------------------------------------------------
    # Preprocessing
    # ------------------------------------------------------------------
    # Normalization method for oceanographic channels: 'minmax' or 'zscore'
    'norm_method': 'minmax',
    # xarray/pandas resample frequency ('1ME' replaces deprecated '1M')
    'resample_freq': '1ME',
}

DATA_DIR   = CONFIG['data_dir']     # convenience alias
bbox_r     = CONFIG['bbox_regional']
dates_full = CONFIG['date_full']
print(f'DATA_DIR      : {DATA_DIR}')
print(f'Regional bbox : {bbox_r}')
print(f'Full dates    : {dates_full}')

Mounted at /content/drive
DATA_DIR      : /content/drive/MyDrive/fishing_project/
Regional bbox : {'lat_min': 0, 'lat_max': 30, 'lon_min': 110, 'lon_max': 140}
Full dates    : {'start': '2014-01-01', 'end': '2024-12-31'}


In [3]:
# CELL: load raw CMEMS NetCDF datasets from Drive
f = CONFIG['files']

print('Loading CMEMS Global Ocean Physics Reanalysis (Wind Velocities)...')
w_physics_ds = xr.open_dataset(DATA_DIR + f['physics_w_nc'])
print(f'  Velocity variables : {list(w_physics_ds.data_vars)}')
print(f'  Shape              : {dict(w_physics_ds.sizes)}')

print('Loading CMEMS Global Ocean Physics Reanalysis (Height & Temp)...')
physics_ds = xr.open_dataset(DATA_DIR + f['physics_ht_nc'])
print(f'  Variables : {list(physics_ds.data_vars)}')
print(f'  Shape     : {dict(physics_ds.sizes)}')

print('Loading CMEMS Global Ocean Biogeochemistry Hindcast...')
bgc_ds = xr.open_dataset(DATA_DIR + f['bgc_src_nc'])
print(f'  Variables : {list(bgc_ds.data_vars)}')
print(f'  Shape     : {dict(bgc_ds.sizes)}')

Loading CMEMS Global Ocean Physics Reanalysis (Wind Velocities)...
  Velocity variables : ['uo', 'vo']
  Shape              : {'time': 132, 'depth': 5, 'latitude': 361, 'longitude': 360}
Loading CMEMS Global Ocean Physics Reanalysis (Height & Temp)...
  Variables : ['zos', 'thetao']
  Shape     : {'time': 132, 'latitude': 361, 'longitude': 360, 'depth': 5}
Loading CMEMS Global Ocean Biogeochemistry Hindcast...
  Variables : ['chl', 'nppv']
  Shape     : {'time': 132, 'depth': 5, 'latitude': 121, 'longitude': 121}


In [4]:
# CELL: load AIS fishing effort CSVs (flat folder, regional bbox-filtered)
# Reads only ais_use_cols; skips files outside the date range by filename;
# filters rows to the regional bbox before concat -- never loads global data.
r  = CONFIG['bbox_regional']
dt = CONFIG['date_full']
AIS_ROOT = DATA_DIR + 'ais_fishing/'

def load_ais_filtered(ais_root, date_start, date_end,
                      lat_min, lat_max, lon_min, lon_max, use_cols):
    '''Load GFW AIS monthly CSVs filtered to a bounding box and date range.'''
    start  = pd.Timestamp(date_start)
    end    = pd.Timestamp(date_end)
    chunks = []

    # All CSVs are flat in one folder -- no year subfolders
    all_files = sorted(glob.glob(os.path.join(ais_root, '*.csv')))
    print(f'Total CSVs found in folder: {len(all_files)}')

    for fp in all_files:
        fname = os.path.basename(fp)
        # Extract date from filename: fleet-monthly-csvs-10-v3-YYYY-MM-DD.csv
        try:
            file_date = pd.Timestamp(fname[-14:-4])
        except Exception:
            continue
        # Skip files outside date range without opening them
        if not (start <= file_date <= end):
            continue

        try:
            df = pd.read_csv(
                fp,
                usecols=use_cols,
                dtype={
                    'cell_ll_lat':   'float32',
                    'cell_ll_lon':   'float32',
                    'fishing_hours': 'float32',
                }
            )
            # Filter to bbox immediately (drops ~98% of rows)
            mask = (
                (df['cell_ll_lat'] >= lat_min) & (df['cell_ll_lat'] <  lat_max) &
                (df['cell_ll_lon'] >= lon_min) & (df['cell_ll_lon'] <  lon_max)
            )
            filtered = df[mask]
            if not filtered.empty:
                chunks.append(filtered)
                n_rows = len(filtered)
                print(f'  {fname[-14:-4]}: {n_rows:,} rows in bbox')
        except Exception as e:
            print(f'  Skipped {fname}: {e}')

    if not chunks:
        raise ValueError('No AIS data found for the given bbox / date range!')

    ais_df = pd.concat(chunks, ignore_index=True)
    ais_df['date']       = pd.to_datetime(ais_df['date'])
    ais_df['year_month'] = ais_df['date'].dt.to_period('M')
    return ais_df


start_yr = dt['start'][:4]
end_yr   = dt['end'][:4]
print(f'Loading AIS CSVs ({start_yr}-{end_yr}, regional bbox)...')
ais_df = load_ais_filtered(
    AIS_ROOT,
    dt['start'], dt['end'],
    r['lat_min'], r['lat_max'], r['lon_min'], r['lon_max'],
    CONFIG['ais_use_cols']
)

n_records    = len(ais_df)
ais_min_date = ais_df['date'].min().date()
ais_max_date = ais_df['date'].max().date()
n_months     = ais_df['year_month'].nunique()
print(f'AIS records in regional bbox : {n_records:,}')
print(f'Date range                   : {ais_min_date} to {ais_max_date}')
print(f'Unique months                : {n_months}')

Loading AIS CSVs (2014-2024, regional bbox)...
Total CSVs found in folder: 132
  2014-01-01: 10,933 rows in bbox
  2014-02-01: 10,793 rows in bbox
  2014-03-01: 13,209 rows in bbox
  2014-04-01: 14,541 rows in bbox
  2014-05-01: 17,068 rows in bbox
  2014-06-01: 8,333 rows in bbox
  2014-07-01: 10,944 rows in bbox
  2014-08-01: 14,027 rows in bbox
  2014-09-01: 13,795 rows in bbox
  2014-10-01: 14,701 rows in bbox
  2014-11-01: 13,691 rows in bbox
  2014-12-01: 14,403 rows in bbox
  2015-01-01: 15,224 rows in bbox
  2015-02-01: 11,987 rows in bbox
  2015-03-01: 16,726 rows in bbox
  2015-04-01: 19,434 rows in bbox
  2015-05-01: 19,287 rows in bbox
  2015-06-01: 9,985 rows in bbox
  2015-07-01: 13,313 rows in bbox
  2015-08-01: 15,643 rows in bbox
  2015-09-01: 12,416 rows in bbox
  2015-10-01: 9,229 rows in bbox
  2015-11-01: 14,054 rows in bbox
  2015-12-01: 12,989 rows in bbox
  2016-01-01: 15,030 rows in bbox
  2016-02-01: 15,947 rows in bbox
  2016-03-01: 22,594 rows in bbox
  2016

In [5]:
# CELL: merge physics datasets into a single xarray Dataset
# Ensure consistent dimension order before merging
w_physics_ds = w_physics_ds.transpose('time', 'depth', 'latitude', 'longitude')
physics_ds   = physics_ds.transpose('time', 'depth', 'latitude', 'longitude')

print('Merging physics datasets (uo, vo, zos, thetao)...')
physics_ds = xr.merge([w_physics_ds, physics_ds])

combined_vars  = list(physics_ds.data_vars)
combined_shape = dict(physics_ds.sizes)
print(f'Combined variables : {combined_vars}')
print(f'Combined shape     : {combined_shape}')

Merging physics datasets (uo, vo, zos, thetao)...
Combined variables : ['uo', 'vo', 'zos', 'thetao']
Combined shape     : {'time': 132, 'depth': 5, 'latitude': 361, 'longitude': 360}


In [6]:
# CELL: time range verification -- confirm all three sources share the configured span
print('=== Time Range Verification ===')
phys_min = physics_ds.time.min().values
phys_max = physics_ds.time.max().values
bgc_min  = bgc_ds.time.min().values
bgc_max  = bgc_ds.time.max().values
ais_min  = ais_df['date'].min()
ais_max  = ais_df['date'].max()

print(f'Physics data : {phys_min} to {phys_max}')
print(f'BGC data     : {bgc_min} to {bgc_max}')
print(f'AIS data     : {ais_min} to {ais_max}')

expected_start = pd.Timestamp(CONFIG['date_full']['start'])
assert pd.Timestamp(str(phys_min)) >= expected_start, 'Physics start out of configured range!'
assert pd.Timestamp(str(bgc_min))  >= expected_start, 'BGC start out of configured range!'
assert ais_min                      >= expected_start, 'AIS start out of configured range!'
print('All three sources cover the configured date range.')

=== Time Range Verification ===
Physics data : 2014-01-01T00:00:00.000000000 to 2024-12-01T00:00:00.000000000
BGC data     : 2014-01-01T00:00:00.000000000 to 2024-12-01T00:00:00.000000000
AIS data     : 2014-01-01 00:00:00 to 2024-12-01 00:00:00
All three sources cover the configured date range.


In [7]:
# CELL: save outputs -- regional NetCDFs + AIS Parquet for Notebook 02
# These files are the sole inputs to 02_preprocessing.ipynb.
f = CONFIG['files']

physics_nc_name  = f['physics_nc']
bgc_nc_name      = f['bgc_nc']
ais_parquet_name = f['ais_parquet']
ais_csv_gz_name  = f['ais_csv_gz']

# 1. Save merged raw physics and BGC regional datasets
physics_ds.to_netcdf(DATA_DIR + physics_nc_name)
bgc_ds.to_netcdf(DATA_DIR + bgc_nc_name)
print(f'Saved physics : {physics_nc_name}')
print(f'Saved BGC     : {bgc_nc_name}')

# 2. Save regional AIS DataFrame as Parquet (fast read, preserves dtypes)
try:
    ais_df.to_parquet(DATA_DIR + ais_parquet_name, index=False)
    print(f'Saved AIS     : {ais_parquet_name}')
except Exception:
    # Fallback to compressed CSV if pyarrow/fastparquet not available
    ais_df.to_csv(DATA_DIR + ais_csv_gz_name, compression='gzip', index=False)
    print(f'Saved AIS     : {ais_csv_gz_name} (compressed CSV fallback)')

print('Notebook 01 complete. Run 02_preprocessing.ipynb next.')

Saved physics : physics_raw_region.nc
Saved BGC     : bgc_raw_region.nc
Saved AIS     : ais_raw_region.parquet
Notebook 01 complete. Run 02_preprocessing.ipynb next.
